# Article Classifier

In [45]:
#imports
import pandas as pd
from tqdm import tqdm
from nltk.tokenize import RegexpTokenizer
import re
from IPython.display import display, HTML
import os
from html import escape


folder_path = 'elections_data_with_text.csv'

In [52]:
df = pd.read_csv(folder_path)
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text,party,cleaned_text
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...,Aanpassen aan \nde draagkracht \nvan de aarde\...,De Groenen,Aanpassen aan de draagkracht van de aarde DE G...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...,Verkiezingsprogramma\n2025 - 2029\n\n Dankwoor...,BoerBurgerBeweging,Verkiezingsprogramma 2025 - 2029 Dankwoord Lie...
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...,Bouwen op\nvertrouwen.\n\nOnze keuzes voor\nee...,CDA,Bouwen op vertrouwen. Onze keuzes voor een fat...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...,eiligheid\n\n V\ne\nz\nn\nO\n\nOnze Gr\n\ne\nn...,JA21,eiligheid V e z n O Onze Gr e n z e n Onze Wel...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h...",Verkiezingsprogramma Piratenpartij \nNederland...,Piratenpartij,Verkiezingsprogramma Piratenpartij Nederland 2...


In [17]:
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|LLaMA|openai|kunstmatige intelligentie systeem*|intelligente algoritme*|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drone|drones"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord"
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}

_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='cleaned_text'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    # print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='cleaned_text'):

    print('test')

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


In [22]:
classified = ai_classification(df, title_col='title', body_col='cleaned_text')  # populates new columns on df


test


100%|██████████| 333/333 [00:06<00:00, 49.44it/s] 


In [23]:
#check the counts of climate-related articles
print(classified['ai_related'].value_counts())

ai_related
no     242
yes     91
Name: count, dtype: int64


In [36]:
import re
import pandas as pd
from IPython.display import display, HTML

def extract_highlighted_snippets(text, keywords, window=5, max_snippets=5):
    """
    Return HTML with up to `max_snippets` snippets from `text`, each containing a
    matched keyword and `window` words of context on both sides.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Build regex for keywords
    pattern = r'\b(?:' + '|'.join(re.escape(w) for w in keywords) + r')\b'
    flags = re.IGNORECASE

    # All keyword matches (as character spans)
    matches = list(re.finditer(pattern, text, flags))
    if not matches:
        return ""

    # Precompute word spans to translate char positions -> word indices
    word_spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    def word_index_for_pos(pos):
        # Small linear search is fine for typical lengths; could be bisected if huge
        for i, (s, e) in enumerate(word_spans):
            if s <= pos < e:
                return i
        return None

    # Build context windows (as char spans), merging overlaps
    windows = []
    for m in matches:
        wi = word_index_for_pos(m.start())
        if wi is None:
            continue
        start_w = max(0, wi - window)
        end_w   = min(len(word_spans), wi + window + 1)
        start_c = word_spans[start_w][0]
        end_c   = word_spans[end_w - 1][1]

        if windows and start_c <= windows[-1][1]:
            # merge with previous overlapping window
            windows[-1] = (windows[-1][0], max(windows[-1][1], end_c))
        else:
            windows.append((start_c, end_c))

    # Limit number of snippets
    windows = windows[:max_snippets]

    # Build HTML snippets with highlighting
    pieces = []
    for s, e in windows:
        chunk = text[s:e]
        chunk = re.sub(pattern,
                       lambda mm: '<span style="font-weight:bold;color:green;">{}</span>'.format(mm.group(0)),
                       chunk, flags=flags)
        prefix = '...' if s > 0 else ''
        suffix = '...' if e < len(text) else ''
        pieces.append(prefix + chunk + suffix)

    return ' '.join(pieces)


# --- Keep your original highlighter for titles (optional) ---
def highlight_keywords(text, keywords):
    if not isinstance(keywords, (set, list)):
        raise ValueError("Keywords must be a set or list")
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in keywords) + r')\b'
    return re.sub(
        pattern,
        lambda m: f'<span style="font-size: 1.5em; font-weight: bold; color: green;">{m.group(0)}</span>',
        text or "",
        flags=re.IGNORECASE
    )


def inspect_ai_related(df, keywords, num_samples=5, window=5, max_snippets=5):
    """
    Displays random samples of AI-related rows with keyword-highlighted snippets.
    """
    subset = df[df['ai_related'] == 'yes']
    if subset.empty:
        display(HTML("<p><em>No AI-related rows found.</em></p>"))
        return

    n = min(len(subset), num_samples)
    samples = subset.sample(n=n, random_state=40)

    for _, row in samples.iterrows():
        title = row.get('title') or row.get('title') or ''
        body  = row.get('cleaned_text')  or ''

        highlighted_title = highlight_keywords(str(title), keywords)
        # Only show short snippets around matches in the body
        highlighted_body_snippets = extract_highlighted_snippets(str(body), keywords,
                                                                 window=20,
                                                                 max_snippets=max_snippets)

        display(HTML(f'<h2>Title: {highlighted_title}</h2>'))
        if highlighted_body_snippets:
            display(HTML(f'<p>{highlighted_body_snippets}</p>'))
        else:
            display(HTML('<p><em>No keyword hits in body.</em></p>'))
        display(HTML('<hr>'))


In [37]:
import ast

def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

classified['matched_keywords_all'] = classified.apply(
    lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords_all']),
    axis=1
)
classified['matched_keywords_all'].iloc[0]

['drones']

In [27]:
#drop rows
df = classified.drop(columns=['company_hits', 'matched_keywords_title', 'matched_keywords_body', 'n_hits_title_total', 'n_hits_body_total'])

In [41]:


def inspect_ai_related(df, body='relevant_section',  num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """
    
    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', body}.issubset(df.columns):
        missing = {'title', body} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)
    
    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        
        ai_val = row['ai_related']
        title = row['title']
        body_text = row[body]  # ✅ 'body' parameter stays intact

         # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples
    

In [29]:
import re
import ast
import pandas as pd

def _norm_matched_keywords(val):
    """
    Normalize matched_keywords_all into a list[str].
    Accepts: list/set/tuple, stringified list (e.g. "['ai','ml']"),
             or comma-separated string ("ai, machine learning").
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, set, tuple)):
        kws = list(val)
    elif isinstance(val, str):
        s = val.strip()
        if not s:
            return []
        # Try to parse stringified list
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, set, tuple)):
                kws = list(parsed)
            else:
                # fallback: treat as comma-separated
                kws = [x.strip() for x in s.split(",") if x.strip()]
        except Exception:
            kws = [x.strip() for x in s.split(",") if x.strip()]
    else:
        return []

    # clean, dedupe, keep phrases
    out = []
    seen = set()
    for k in kws:
        if not isinstance(k, str):
            continue
        k2 = k.strip()
        if not k2:
            continue
        kl = k2.lower()
        if kl not in seen:
            seen.add(kl)
            out.append(k2)
    return out

def _extract_windows(text, keywords, window=5, max_snippets=5):
    """
    Build a plain-text section composed of ±window words around each keyword hit.
    Returns: (section_text, spans, n_matches)
    """
    if not isinstance(text, str) or not text.strip():
        return "", [], 0
    keywords = _norm_matched_keywords(keywords)
    if not keywords:
        return "", [], 0

    # Safer "word boundary" for phrases: (?<!\w) ... (?!\w)
    # (handles multi-word phrases and punctuation better than \b ... \b)
    alts = [re.escape(k) for k in keywords]
    pattern = r'(?<!\w)(?:' + '|'.join(alts) + r')(?!\w)'
    flags = re.IGNORECASE

    matches = list(re.finditer(pattern, text, flags))
    if not matches:
        return "", [], 0

    # Word spans to take ±window words
    word_spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    def word_index_for_pos(pos):
        # small linear scan is fine for typical lengths
        for i, (s, e) in enumerate(word_spans):
            if s <= pos < e:
                return i
        # if match starts on whitespace, snap to next word
        for i, (s, e) in enumerate(word_spans):
            if pos < e:
                return i
        return len(word_spans) - 1 if word_spans else 0

    spans = []
    for m in matches:
        wi = word_index_for_pos(m.start())
        start_w = max(0, wi - window)
        end_w   = min(len(word_spans), wi + window + 1)
        start_c = word_spans[start_w][0]
        end_c   = word_spans[end_w - 1][1]

        if spans and start_c <= spans[-1][1]:
            spans[-1] = (spans[-1][0], max(spans[-1][1], end_c))
        else:
            spans.append((start_c, end_c))

    spans = spans[:max_snippets]

    parts = []
    for i, (s, e) in enumerate(spans):
        prefix = '...' if (i == 0 and s > 0) else ''
        suffix = '...' if (i == len(spans)-1 and e < len(text)) else ''
        parts.append(prefix + text[s:e].strip() + suffix)

    section_text = ' '.join(parts)
    return section_text, spans, len(matches)

def add_relevant_section(df, body_col='body', matched_col='matched_keywords_all',
                         window=5, max_snippets=5):
    """
    Adds:
      - relevant_section (plain text)
      - relevant_section_spans (char spans)
      - n_keyword_matches
      - relevant_len_chars
      - relevant_len_words
    """
    results = df.apply(
        lambda r: _extract_windows(
            str(r[body_col]) if pd.notna(r.get(body_col)) else "",
            r.get(matched_col),
            window,
            max_snippets
        ),
        axis=1
    )
    df['relevant_section']       = results.map(lambda t: t[0])
    # df['relevant_section_spans'] = results.map(lambda t: t[1])
    # df['n_keyword_matches']      = results.map(lambda t: t[2])
    # df['relevant_len_chars']     = df['relevant_section'].str.len()
    # df['relevant_len_words']     = df['relevant_section'].str.count(r'\b\w+\b')
    return df


In [39]:
df_relevant = add_relevant_section(df, body_col='cleaned_text', matched_col='matched_keywords_all', window=50)
df_relevant

,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text,party,cleaned_text,ai_related,matched_keywords_all,relevant_section
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...,Aanpassen aan \nde draagkracht \nvan de aarde\...,De Groenen,Aanpassen aan de draagkracht van de aarde DE G...,no,[drones],...rights in the occupied Palestinian territor...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...,Verkiezingsprogramma\n2025 - 2029\n\n Dankwoor...,BoerBurgerBeweging,Verkiezingsprogramma 2025 - 2029 Dankwoord Lie...,yes,"[ai, drone, drones]",...ruimte laat voor urgentie of realiteitszin....
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...,Bouwen op\nvertrouwen.\n\nOnze keuzes voor\nee...,CDA,Bouwen op vertrouwen. Onze keuzes voor een fat...,yes,"[ai, algoritmes, gpt, kunstmatige intelligentie]",...Als mantelzorger of reservist. Waar mensen ...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...,eiligheid\n\n V\ne\nz\nn\nO\n\nOnze Gr\n\ne\nn...,JA21,eiligheid V e z n O Onze Gr e n z e n Onze Wel...,yes,"[ai, drones, kunstmatige intelligentie]",...gemiddelden waarmee men uitsluitend op papi...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h...",Verkiezingsprogramma Piratenpartij \nNederland...,Piratenpartij,Verkiezingsprogramma Piratenpartij Nederland 2...,yes,"[facebook, google, ai, algoritme, algoritmen, ...",...betaalbare woningen goede zorg en onderwijs...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,https://dnpprepo.ub.rug.nl/10852/,Voor Nederland. Verkiezingsprogramma 2017-2021,2017,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/10852/7/VNL%20verki...,VNL (2017) Voor Nederland. Verkiezingsprogram...,cc d e m o c r a t i e \nw\n\n M I N D E\n\...,VNL,cc d e m o c r a t i e w M I N D E R! = 0 ^ t ...,no,[drones],...een vliegdekschip. • Meer fregatten mijnenj...
329,https://dnpprepo.ub.rug.nl/86333/,Waardevol voor Westerwolde,2017,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/86333/1/CU_Westerwo...,ChristenUnie (2017) Waardevol voor Westerwold...,Programma 2017 \n\nWaardevol voor Westerwolde ...,ChristenUnie,Programma 2017 Waardevol voor Westerwolde de C...,no,[],
330,https://dnpprepo.ub.rug.nl/86457/,Westerwolde,2017,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/86457/1/VVD_Westerw...,VVD (2017) Westerwolde. [Verkiezingsprogramm...,Westerwolde \n\nDe speerpunten van VVD Westerw...,VVD,Westerwolde De speerpunten van VVD Westerwolde...,no,[],
331,https://dnpprepo.ub.rug.nl/10862/,Zeker Nederland,2017,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text', 'text', 'text', 'text']","['nl', 'nl', 'nl', 'nl', 'nl']",https://dnpprepo.ub.rug.nl/10862/1/VVD-VerkPro...,VVD (2017) Zeker Nederland. [Verkiezingsprog...,ZEKER\nNEDERLAND\n\nVVD verkiezingsprogramma \...,VVD,ZEKER NEDERLAND VVD verkiezingsprogramma 2017-...,yes,"[drones, kunstmatige intelligentie]",...normaal vinden. Dat het hier normaal is dat...


In [47]:
# filter for only ai related
df_ai_related = df_relevant[df_relevant['ai_related'] == 'yes']

In [49]:
df_ai_related.to_csv('elections_data_ai_related_with_relevant_sections.csv', index=False)

In [50]:
df_relevant.to_csv('elections_data_with_relevant_sections.csv', index=False)

In [48]:
# program = "all" to display from all programs, or specify a program like "jinek"
inspect_ai_related(df_ai_related, body = 'relevant_section', num_samples=5, random_state=24)

Displayed 5 random articles (with row-specific matched keywords highlighted).


,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text,party,cleaned_text,ai_related,matched_keywords_all,relevant_section
72,https://dnpprepo.ub.rug.nl/87710/,Ruimte geven. Grenzen stellen.,2023,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text', 'text']","['nl', 'nl', 'nl']",https://dnpprepo.ub.rug.nl/87710/7/VVD%20Verki...,VVD (2023) Ruimte geven. Grenzen stellen. [V...,Ruimte geven.\nGrenzen stellen.\n\nKeuzes voor...,VVD,Ruimte geven. Grenzen stellen. Keuzes voor een...,yes,"[ai, algoritmen, artificial intelligence, arti...",...Er zijn goede voorbeelden van nieuwe samenw...
142,https://dnpprepo.ub.rug.nl/12186/,Voor verandering,2019,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/12186/1/GroenLinks%...,GroenLinks (2019) Voor verandering. [Verkiez...,VOOR \nVERANDERING\n\nVERKIEZINGSPROGRAMMA \nE...,GroenLinks,VOOR VERANDERING VERKIEZINGSPROGRAMMA EUROPESE...,yes,"[apple, facebook, google, uber, algoritmen, al...",...waarden. Europa beschermt de democratie de ...
52,https://dnpprepo.ub.rug.nl/87782/,Concept verkiezingsprogramma Tweede kamer 2023...,2023,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/87782/1/Splinter%20...,Splinter (2023) Concept verkiezingsprogramma ...,CONCEPT VERKIEZINGSPROGRAMMA \n\nTWEEDE KAMER...,Splinter,CONCEPT VERKIEZINGSPROGRAMMA TWEEDE KAMER 2023...,yes,"[microsoft, ai, algoritme, algoritmen, algorit...",...EUROPA........................................
30,https://dnpprepo.ub.rug.nl/87872/,BBBeter Nederland - Er staat veel op het spel,2024,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/87872/1/BBB%20Verki...,BoerBurgerBeweging (2024) BBBeter Nederland -...,Er staat veel \nop het spel\n\nVerkiezingsprog...,BoerBurgerBeweging,Er staat veel op het spel Verkiezingsprogramma...,yes,"[ai, algoritmes, artificial intelligence, kuns...",...langs de nationale grenzen. De Europese Com...
117,https://dnpprepo.ub.rug.nl/86168/,Verkiezingsprogramma 2021,2021,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/86168/1/Zetelgenoot...,Het Zetel Genootschap (2021) Verkiezingsprogr...,"Vierde conceptversie, gepubliceerd 30 november...",Het Zetel Genootschap,Vierde conceptversie gepubliceerd 30 november ...,yes,"[google, uber, artificiële intelligentie]",...journalistiek opinie maken amusement en onw...


In [ ]:
##create random sample of X articles
#n_per_outlet = 30 // df['outlet'].nunique()

#balanced_sample1 = (
#    df.groupby('outlet', group_keys=False)
#      .apply(lambda x: x.sample(n=n_per_outlet, random_state=42))
#      .sample(frac=1, random_state=42)  # shuffle
#      .reset_index(drop=True)
#)

In [ ]:
#balanced_sample1.to_csv('sample.csv')

In [22]:
# Save the classified data
df.to_csv('party_programs_classified.csv')